In [8]:
import pandas as pd
df=pd.read_excel('../experimental_data_for_model_without_xichuxiang.xlsx',index_col = 0)
display(df)
correct_order = ['Grain Size', 'Calculated Solid Solution', 'Calculated Grain Boundary', 'Calculated Yield Strength', 'Yield_Strength', 'Tensile_Strength (UTS)']
# 获取DataFrame的列名列表
columns = df.columns.tolist()
# 将倒数第六列的特征按照指定顺序调整到正确位置
corrected_columns = columns[:-6] + correct_order 
# 重新排列DataFrame的列
df = df[corrected_columns]
display(df)
df.to_excel('experimental_data_for_model_without_xichuxiang.xlsx',index = True)


,MagpieData minimum Number,MagpieData maximum Number,MagpieData range Number,MagpieData mean Number,MagpieData avg_dev Number,MagpieData mode Number,MagpieData minimum MendeleevNumber,MagpieData maximum MendeleevNumber,MagpieData range MendeleevNumber,MagpieData mean MendeleevNumber,...,frac s valence electrons,frac p valence electrons,frac d valence electrons,frac f valence electrons,Grain Size,屈服强度,抗拉强度 (UTS),Calculated Solid Solution,Calculated Grain Boundary,Calculated Yield Strength
0,12,39,27,12.761734,1.480487,12,12,68,56,66.420108,...,0.98609,0,0.01391,0,12.9,103,189,74.139496,66.323806,155.463302
1,12,39,27,12.761734,1.480487,12,12,68,56,66.420108,...,0.98609,0,0.01391,0,12.9,105,190,74.139496,66.323806,155.463302
2,12,39,27,12.761734,1.480487,12,12,68,56,66.420108,...,0.98609,0,0.01391,0,12.9,111,192,74.139496,66.323806,155.463302


,MagpieData minimum Number,MagpieData maximum Number,MagpieData range Number,MagpieData mean Number,MagpieData avg_dev Number,MagpieData mode Number,MagpieData minimum MendeleevNumber,MagpieData maximum MendeleevNumber,MagpieData range MendeleevNumber,MagpieData mean MendeleevNumber,...,frac s valence electrons,frac p valence electrons,frac d valence electrons,frac f valence electrons,Grain Size,Calculated Solid Solution,Calculated Grain Boundary,Calculated Yield Strength,屈服强度,抗拉强度 (UTS)
0,12,39,27,12.761734,1.480487,12,12,68,56,66.420108,...,0.98609,0,0.01391,0,12.9,74.139496,66.323806,155.463302,103,189
1,12,39,27,12.761734,1.480487,12,12,68,56,66.420108,...,0.98609,0,0.01391,0,12.9,74.139496,66.323806,155.463302,105,190
2,12,39,27,12.761734,1.480487,12,12,68,56,66.420108,...,0.98609,0,0.01391,0,12.9,74.139496,66.323806,155.463302,111,192


In [20]:
import pandas 
df=pd.read_excel('experimental_data_for_model_without_xichuxiang.xlsx',index_col = 0)
import joblib
X_kl = df.drop(columns=['Yield_Strength', 'Tensile_Strength (UTS)'])
print(X_kl.shape)
y_kl = df['Tensile_Strength (UTS)']
model=joblib.load('../抗拉强度/models/xgboost/xgboost1_new.pkl')
print(model.predict(X_kl))

X_qf = df.drop(columns=['Yield_Strength', 'Tensile_Strength (UTS)'])
print(X_qf.shape)
y_qf = df['Yield_Strength']
model=joblib.load('../屈服强度/models/gboost/gboost1_new.pkl')
print(model.predict(X_qf))

(3, 279)
[203.33675 203.33675 203.33675]
(3, 279)
[242.9486072 242.9486072 242.9486072]


In [22]:
import joblib
import numpy as np
import pandas as pd
import os

# 定义文件夹路径
folder_path_qf = "../屈服强度/models"
folder_path_kl = '../抗拉强度/models'

# 定义模型权重
weights_qf = {'gboost': 0.35 / 2, 'xgboost': 0.35 / 2, 'knn': 0.3 / 2}  # 注意所有模型系数和为1
weights_kl = {'rf': 0.2 / 2, 'xgboost': 0.3 / 2, 'knn': 0.5 / 2}  # 注意所有模型系数和为1

# 读取数据
df_qf = pd.read_excel('../experimental_data_for_model_without_xichuxiang.xlsx', index_col=0)
df_kl = pd.read_excel('../experimental_data_for_model_without_xichuxiang.xlsx', index_col=0)

# 分割特征和目标变量
X_qf = df_qf.drop(columns=['Yield_Strength', 'Tensile_Strength (UTS)'])
y_qf = df_qf['Yield_Strength']
X_kl = df_kl.drop(columns=['Yield_Strength', 'Tensile_Strength (UTS)'])
y_kl = df_kl['Tensile_Strength (UTS)']

# 初始化预测结果
predictions_fold_qf = np.zeros(len(X_qf))
predictions_fold_kl = np.zeros(len(X_kl))

# 遍历QF模型进行加权（仅使用第二个和第四个模型）
for model_name, weight in weights_qf.items():
    for i in [2, 4]:  # 只使用第二个和第四个模型
        model_path = f'{folder_path_qf}/{model_name}/{model_name}{i}_new.pkl'
        if os.path.exists(model_path):
            model = joblib.load(model_path)
            predictions_fold_qf += model.predict(X_qf) * weight
        else:
            print(f"Model file {model_path} does not exist.")

# 遍历KL模型进行加权（仅使用第二个和第四个模型）
for model_name, weight in weights_kl.items():
    for i in [2, 4]:  # 只使用第二个和第四个模型
        model_path = f'{folder_path_kl}/{model_name}/{model_name}{i}_new.pkl'
        if os.path.exists(model_path):
            model = joblib.load(model_path)
            predictions_fold_kl += model.predict(X_kl) * weight
        else:
            print(f"Model file {model_path} does not exist.")

# 打印预测结果和误差
for i in range(len(predictions_fold_qf)):
    qf_error = predictions_fold_qf[i] - y_qf.iloc[i]
    kl_error = predictions_fold_kl[i] - y_kl.iloc[i]
    print(f"Sample {i + 1} - QF Prediction: {predictions_fold_qf[i]}, Actual QF: {y_qf.iloc[i]}, QF Error: {qf_error}")
    print(f"Sample {i + 1} - KL Prediction: {predictions_fold_kl[i]}, Actual KL: {y_kl.iloc[i]}, KL Error: {kl_error}")

# 打印模型评估指标


Sample 1 - QF Prediction: 242.96367630716625, Actual QF: 103, QF Error: 139.96367630716625
Sample 1 - KL Prediction: 302.8133679191159, Actual KL: 189, KL Error: 113.81336791911588
Sample 2 - QF Prediction: 242.96367630716625, Actual QF: 105, QF Error: 137.96367630716625
Sample 2 - KL Prediction: 302.8133679191159, Actual KL: 190, KL Error: 112.81336791911588
Sample 3 - QF Prediction: 242.96367630716625, Actual QF: 111, QF Error: 131.96367630716625
Sample 3 - KL Prediction: 302.8133679191159, Actual KL: 192, KL Error: 110.81336791911588


In [26]:
import joblib
import numpy as np
import pandas as pd
import os
from sklearn.metrics import mean_absolute_error, r2_score,mean_absolute_percentage_error

# 定义文件夹路径
folder_path_qf = "../屈服强度/models"
folder_path_kl = '../抗拉强度/models'

# 定义模型权重
weights_qf = {'gboost': 0.35 / 2, 'xgboost': 0.35 / 2, 'knn': 0.3 / 2}  # 注意所有模型系数和为1
weights_kl = {'rf': 0.2 / 2, 'xgboost': 0.3 / 2, 'knn': 0.5 / 2}  # 注意所有模型系数和为1

# 读取数据
df_qf = pd.read_excel('20240127FULL.xlsx')
df_kl = pd.read_excel('20240127FULL.xlsx')

# 分割特征和目标变量
X_qf = df_qf.drop(columns=['Yield_Strength', 'Tensile_Strength (UTS)', '追踪编号'])
y_qf = df_qf['Yield_Strength']
X_kl = df_kl.drop(columns=['Yield_Strength', 'Tensile_Strength (UTS)', '追踪编号'])
y_kl = df_kl['Tensile_Strength (UTS)']
tracking_ids = df_qf['追踪编号']

# 初始化预测结果
predictions_fold_qf = np.zeros(len(X_qf))
predictions_fold_kl = np.zeros(len(X_kl))

# 遍历QF模型进行加权（仅使用第二个和第四个模型）
for model_name, weight in weights_qf.items():
    for i in [2, 4]:  # 只使用第二个和第四个模型
        model_path = f'{folder_path_qf}/{model_name}/{model_name}{i}_new.pkl'
        if os.path.exists(model_path):
            model = joblib.load(model_path)
            predictions_fold_qf += model.predict(X_qf) * weight
        else:
            print(f"Model file {model_path} does not exist.")

# 遍历KL模型进行加权（仅使用第二个和第四个模型）
for model_name, weight in weights_kl.items():
    for i in [2, 4]:  # 只使用第二个和第四个模型
        model_path = f'{folder_path_kl}/{model_name}/{model_name}{i}_new.pkl'
        if os.path.exists(model_path):
            model = joblib.load(model_path)
            predictions_fold_kl += model.predict(X_kl) * weight
        else:
            print(f"Model file {model_path} does not exist.")

# 计算误差并保存结果
results = pd.DataFrame({
    '追踪编号': tracking_ids,
    '实际屈服强度': y_qf,
    '预测屈服强度': predictions_fold_qf,
    '屈服强度误差': predictions_fold_qf - y_qf,
    '实际抗拉强度': y_kl,
    '预测抗拉强度': predictions_fold_kl,
    '抗拉强度误差': predictions_fold_kl - y_kl
})

# 保存结果到新的Excel文件中
output_file = '20240127FULL_predictions.xlsx'
results.to_excel(output_file, index=False)

# 计算整体预测精度
r2_qf = r2_score(y_qf, predictions_fold_qf)
mae_qf = mean_absolute_error(y_qf, predictions_fold_qf)
mape_qf = mean_absolute_percentage_error(y_qf, predictions_fold_qf)

r2_kl = r2_score(y_kl, predictions_fold_kl)
mae_kl = mean_absolute_error(y_kl, predictions_fold_kl)
mape_kl = mean_absolute_percentage_error(y_kl, predictions_fold_kl)

print(f"QF Model - R2: {r2_qf}, MAE: {mae_qf}, MAPE: {mape_qf}")
print(f"KL Model - R2: {r2_kl}, MAE: {mae_kl}, MAPE: {mape_kl}")

del_num=10
# 删除误差最大的5个样本
top_5_error_indices_qf = results['屈服强度误差'].abs().nlargest(10).index
top_5_error_indices_kl = results['抗拉强度误差'].abs().nlargest(10).index
top_5_error_indices = top_5_error_indices_qf.union(top_5_error_indices_kl)

results_new = results.drop(index=top_5_error_indices)

# 重新计算整体预测精度
r2_qf_new = r2_score(results_new['实际屈服强度'], results_new['预测屈服强度'])
mae_qf_new = mean_absolute_error(results_new['实际屈服强度'], results_new['预测屈服强度'])
mape_qf_new = mean_absolute_percentage_error(results_new['实际屈服强度'], results_new['预测屈服强度'])

r2_kl_new = r2_score(results_new['实际抗拉强度'], results_new['预测抗拉强度'])
mae_kl_new = mean_absolute_error(results_new['实际抗拉强度'], results_new['预测抗拉强度'])
mape_kl_new = mean_absolute_percentage_error(results_new['实际抗拉强度'], results_new['预测抗拉强度'])

print(f"QF Model after removing top 5 errors - R2: {r2_qf_new}, MAE: {mae_qf_new}, MAPE: {mape_qf_new}")
print(f"KL Model after removing top 5 errors - R2: {r2_kl_new}, MAE: {mae_kl_new}, MAPE: {mape_kl_new}")

# 保存新的结果到Excel文件
output_file_new = '20240127FULL_predictions_new.xlsx'
results_new.to_excel(output_file_new, index=False)


QF Model - R2: 0.09726899139349054, MAE: 65.21870781896845, MAPE: 3.889111093516101e+16
KL Model - R2: -0.1434711831963189, MAE: 53.683500875900144, MAPE: 0.2706091445279903
QF Model after removing top 5 errors - R2: 0.5270441112693793, MAE: 40.89377561059324, MAPE: 0.2453993400669441
KL Model after removing top 5 errors - R2: 0.42772273554370677, MAE: 32.30941547938803, MAPE: 0.12946447765463978
